# Setup and Settings

In [ ]:
from pathlib import Path

import pandas as pd

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)

source_path = project_root / r"path/to/file.xlsb"

# Discover Sheets

In [ ]:
if source_path.suffix.lower() == ".csv":
    sheet_names = [source_path.stem]
    row_counts = [len(pd.read_csv(source_path))]
else:
    with pd.ExcelFile(source_path, engine="calamine") as workbook:
        sheet_names = workbook.sheet_names
        row_counts = [len(workbook.parse(sheet_name=name)) for name in sheet_names]
df = pd.DataFrame({"name": sheet_names, "row_count": row_counts})
df

# Load Sheet

In [ ]:
sheet = 0
# CSV files hold one table, so sheet applies only to Excel workbooks.
if source_path.suffix.lower() == ".csv":
    df = pd.read_csv(source_path)
else:
    # pandas-stubs references the unavailable optional pyxlsb workbook type.
    df = pd.read_excel(source_path, sheet_name=sheet, engine="calamine")  # pyright: ignore[reportUnknownMemberType]
df.head(5)

# Profile Columns

In [ ]:
df = (
    pd.DataFrame(
        {
            "data_type": df.dtypes.astype(str),
            "non_null_count": df.count(),
            "null_count": df.isna().sum(),
            "null_percent": df.isna().mean().mul(100).round(1),
            "unique_count": df.nunique(),
        }
    )
    .rename_axis("column")
    .reset_index()
)
df